# Notebook 3 - Flood Detection
**Inputs:** data/processed/sar/YYYY-MM_VV.tif + baseline_VV.tif (from Notebook 02)  
**Outputs:** data/outputs/flood_extent/flood_extent_YYYY-MM.tif (COG, uint8)  

## Method
Change detection: pixels where monthly VV backscatter drops >3 dB below the dry baseline  
are flagged as flooded. Water absorbs SAR signal so flooded areas appear significantly  
darker than the same area in non-flood conditions.

## Output values
- 1 = flooded
- 0 = not flooded
- 255 = nodata (terrain shadow, missing data)

## Strengths
- Change detection is robust to uncalibrated data (relative differences matter)
- 3 dB threshold is well-established in SAR flood literature
- Morphological opening removes isolated noise pixels

## Limitations
- Fixed threshold: optimal value varies by terrain and season
- Dense forest can mask flood signal (double-bounce suppresses backscatter drop)
- Urban areas produce false positives due to specular reflection
- Baseline built from first 3 months may itself contain flood events


In [ ]:
# Clone repo (Colab only)
import os
if not os.path.exists("Floodmaps"):
    !git clone https://github.com/trevmon28/Floodmaps.git
os.chdir("Floodmaps")
print("Working directory:", os.getcwd())


In [ ]:
import subprocess, sys
packages = [
    "rasterio", "rioxarray", "numpy", "scipy",
    "scikit-image", "matplotlib", "pyyaml", "tqdm", "pandas"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("Packages ready.")


In [ ]:
import yaml, os, warnings
import numpy as np
import pandas as pd
import rasterio
from rasterio.shutil import copy as rio_copy
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_opening
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tqdm import tqdm
warnings.filterwarnings("ignore")

with open("config/config.yaml") as f:
    cfg = yaml.safe_load(f)

PROCESSED_DIR = "data/processed/sar"
OUTPUT_DIR = "data/outputs/flood_extent"
CHANGE_THRESHOLD_DB = -3.0
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config loaded.")
print("Change threshold:", CHANGE_THRESHOLD_DB, "dB")


In [ ]:
def write_cog(array, profile, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    tmp = output_path + ".tmp.tif"
    p = profile.copy()
    p.update(driver="GTiff", compress="deflate", tiled=True,
             blockxsize=512, blockysize=512, count=1)
    with rasterio.open(tmp, "w", **p) as dst:
        dst.write(array, 1)
    with rasterio.open(tmp, "r+") as dst:
        dst.build_overviews([2, 4, 8, 16], rasterio.enums.Resampling.nearest)
        dst.update_tags(ns="rio_overview", resampling="nearest")
    rio_copy(tmp, output_path, driver="GTiff", copy_src_overviews=True,
             compress="deflate", tiled=True, blockxsize=512, blockysize=512)
    os.remove(tmp)
    print("  Saved:", output_path)

print("Helpers defined.")


In [ ]:
# Load baseline
baseline_path = f"{PROCESSED_DIR}/baseline_VV.tif"
if not os.path.exists(baseline_path):
    raise FileNotFoundError("Baseline not found - run Notebook 02 with process_all=True first")

with rasterio.open(baseline_path) as src:
    baseline = src.read(1).astype("float32")
    profile = src.profile.copy()

print("Baseline loaded:", baseline.shape)
print("Baseline range:", round(float(np.nanmin(baseline)), 1), "to", round(float(np.nanmax(baseline)), 1), "dB")


In [ ]:
vv_files = sorted([f for f in os.listdir(PROCESSED_DIR)
                   if f.endswith("_VV.tif") and "baseline" not in f])
print("Found", len(vv_files), "monthly VV files:")
for f in vv_files:
    print(" ", f)


In [ ]:
flood_stats = []

for fname in tqdm(vv_files, desc="Detecting floods"):
    month_str = fname[:7]
    out_path = f"{OUTPUT_DIR}/flood_extent_{month_str}.tif"

    if os.path.exists(out_path):
        print("  " + month_str + " already exists - skipping")
        continue

    with rasterio.open(f"{PROCESSED_DIR}/{fname}") as src:
        vv = src.read(1).astype("float32")

    change = vv - baseline
    flood_mask = (change < CHANGE_THRESHOLD_DB).astype("uint8")

    nodata_mask = ~np.isfinite(vv) | ~np.isfinite(baseline)
    flood_mask[nodata_mask] = 255

    valid = flood_mask == 1
    cleaned = binary_opening(valid, structure=np.ones((3, 3)))
    flood_mask[valid & ~cleaned] = 0

    out_profile = profile.copy()
    out_profile.update(dtype="uint8", nodata=255)
    write_cog(flood_mask, out_profile, out_path)

    valid_pixels = (flood_mask != 255).sum()
    flooded_pixels = (flood_mask == 1).sum()
    pct = round(flooded_pixels / valid_pixels * 100, 2) if valid_pixels > 0 else 0
    flood_stats.append({"month": month_str, "flooded_pct": pct, "flooded_px": int(flooded_pixels)})
    print("  " + month_str + ": " + str(pct) + "% flooded (" + str(flooded_pixels) + " pixels)")

print("Flood detection complete.")


In [ ]:
# Summary stats table
if flood_stats:
    df = pd.DataFrame(flood_stats).set_index("month")
else:
    rows = []
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        if not fname.endswith(".tif"):
            continue
        month_str = fname.replace("flood_extent_", "").replace(".tif", "")
        with rasterio.open(f"{OUTPUT_DIR}/{fname}") as src:
            arr = src.read(1)
        valid = (arr != 255).sum()
        flooded = (arr == 1).sum()
        pct = round(flooded / valid * 100, 2) if valid > 0 else 0
        rows.append({"month": month_str, "flooded_pct": pct, "flooded_px": int(flooded)})
    df = pd.DataFrame(rows).set_index("month")

print(df.to_string())


In [ ]:
# Plot most recent flood extent map
flood_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".tif")])
if not flood_files:
    print("No flood extent files found - run detection cell first.")
else:
    latest = flood_files[-1]
    with rasterio.open(f"{OUTPUT_DIR}/{latest}") as src:
        flood = src.read(1)

    cmap = mcolors.ListedColormap(["#d4e6b5", "#1a6faf", "#cccccc"])
    bounds = [-0.5, 0.5, 1.5, 255.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    month_label = latest.replace("flood_extent_", "").replace(".tif", "")
    plt.figure(figsize=(14, 10))
    im = plt.imshow(flood, cmap=cmap, norm=norm)
    cbar = plt.colorbar(im, ticks=[0, 1, 255], shrink=0.6)
    cbar.set_ticklabels(["Not flooded", "Flooded", "Nodata"])
    plt.title("Flood Extent - " + month_label)
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    print("File:", OUTPUT_DIR + "/" + latest)
